# RNA-Seq Processing Pipeline
### Authors: Emily Skates, Stephen Williams

Processes STAR + featureCounts outputs through downstream analyses including DESeq2 differential expression, PCA/PERMANOVA clustering, WGCNA co-expression networks, and Gene Ontology/GSEA enrichment.

## 1. Environment Setup & Global Configuration
Initialises required libraries, sets the working directory, and defines global thresholds for gene filtering, plotting, and differential expression cut-offs.

In [ ]:
setwd("/Users/steve/Library/CloudStorage/GoogleDrive-stev3.w1l@gmail.com/My Drive/Academia/c_collaborations/emilyS/")

required_packages <- c("DESeq2", "ggplot2", "ggvenn", "WGCNA", "vegan", "dplyr", "tidyr", "tibble",
                       "pairwiseAdonis", "glue", "scales", "gtools", "clusterProfiler",
                       "enrichplot", "org.Mm.eg.db", "patchwork")

missing_packages <- required_packages[!(required_packages %in% installed.packages()[, "Package"])]
if (length(missing_packages) > 0) {
    install.packages(missing_packages)
}

suppressWarnings(suppressPackageStartupMessages({
    invisible(lapply(required_packages, library, character.only = TRUE))
}))

config <- list(
    min_gene_copies = 10,
    min_samples_per_cond = 1, 
    num_pca_genes = 500,
    num_network_genes = 100,
    log2_fc_thresh = 0.5, 
    padj_thresh = 0.05,
    pval_thresh = 0.05
)

design_type <- 1

options(repr.plot.width = 8, repr.plot.height = 4)

## 2. Data Loading & DESeq2 Initialisation
Compiles the individual gene count files into a unified matrix, defines the sample metadata (including the nested subject design), and initialises the `DESeqDataSet`.

### Understanding the Experimental Design Formula

* **Design type 1, only label state.**
* **Design type 2, only pulldown state (optional include sampleNum)$^*$.**
* **Design type 3, all factors included.**

The DESeq2 design formula breaks down the total variance in gene expression into specific biological, experimental, and technical components:

$$\text{Design Matrix} \sim \text{LabelingState} + \text{LabelingState:SubjectId} + \text{AssayType} + \text{LabelingState:AssayType}$$

* $(1,3)$ **`LabelingState`**: Controls for the baseline difference between the labelled and unlabelled conditions (e.g., global transcriptomic shifts caused by the metabolic labelling itself).
* $(2^*,3)$ **`LabelingState:SubjectId`**: Accounts for individual biological variation by nesting the samples strictly within their respective labelling groups. This captures the paired relationship between an individual sample's Input and Pull-Down samples, preventing sample-to-sample baseline differences from muddying the results.
* $(2,3)$ **`AssayType`**: Measures the non-specific background enrichment. It captures the genes that naturally stick to the beads or tube during a pull-down even when no label is present:
  $$\text{Unlabelled Pull-Down} - \text{Unlabelled Input}$$
* $(3)$ **`LabelingState:AssayType`**: The interaction term between label/pulldown state that isolates the **True Enrichment**. It calculates the "difference of differences," subtracting non-specific background stickiness from the labelled enrichment to find genes specifically captured by the assay:

$$\text{True Target Enrichment} = (\text{Labelled Pull-Down} - \text{Labelled Input}) - (\text{Unlabelled Pull-Down} - \text{Unlabelled Input})$$

In [ ]:
count_files <- gtools::mixedsort(Sys.glob("o_outputs/sample_*/gene_counts.txt"))

# 1. Compile a unified master matrix for all 12 samples first
master_counts <- read.table(count_files[1], header = TRUE, skip = 1, stringsAsFactors = FALSE)[, 1, drop = FALSE]
colnames(master_counts) <- "Geneid"
for (file in count_files) {
    sample_id <- basename(dirname(file))
    master_counts[[sample_id]] <- read.table(file, header = TRUE, skip = 1)[, 7]
}
row.names(master_counts) <- master_counts$Geneid
master_counts$Geneid <- NULL

# 2. Define global metadata mapping all 12 samples
master_col_data <- data.frame(
    LabelingState = factor(rep(c("unlab", "lab", "unlab", "lab"), each = 3), levels = c("unlab", "lab")),
    AssayType = factor(rep(c("pull_down", "input"), each = 6), levels = c("input", "pull_down")),
    SubjectId = factor(rep(c("Subj1", "Subj2", "Subj3"), times = 4)),
    row.names = colnames(master_counts)
)

# 3. Subset data and establish the design matrix
if (design_type == 1) {
    # Design 1: Only pull-down samples (1-6), comparing unlabelled vs labelled
    col_data <- master_col_data[1:6, ]
    count_matrix <- master_counts[, rownames(col_data)]
    dds <- DESeqDataSetFromMatrix(
        countData = count_matrix,
        colData = col_data,
        design = ~ LabelingState
    )

} else if (design_type == 2) {
    # Design 2: Only labelled samples, comparing input vs pull-down
    # Samples 4-6 are labelled pull-down, 10-12 are labelled input
    col_data <- master_col_data[c(4:6, 10:12), ]
    count_matrix <- master_counts[, rownames(col_data)]
    
    dds <- DESeqDataSetFromMatrix(
        countData = count_matrix,
        colData = col_data,
        design = ~ SubjectId + AssayType
    )

} else if (design_type == 3) {
    # Design 3: Full interaction model across all 12 samples
    col_data <- master_col_data
    count_matrix <- master_counts
    dds <- DESeqDataSetFromMatrix(
        countData = count_matrix,
        colData = col_data,
        design = ~ LabelingState + LabelingState:SubjectId + AssayType + LabelingState:AssayType
    )

} else if (design_type == 4) {
    # Design 4: Flat group model mimicking the exploratory R script across all 12 samples
    col_data <- master_col_data
    # Concatenate factors into a single 'Group' variable
    col_data$Group <- factor(paste(col_data$LabelingState, col_data$AssayType, sep = "_"))
    
    # Explicitly set the unlabelled pull-down as the reference baseline
    col_data$Group <- relevel(col_data$Group, ref = "unlab_pull_down") 
    
    count_matrix <- master_counts
    dds <- DESeqDataSetFromMatrix(
        countData = count_matrix,
        colData = col_data,
        design = ~ Group
    )
    
    } else {
    stop("Invalid design type. Please choose 1, 2, or 3.")
}

# Apply global thresholds and run VST
keep_genes <- rowSums(counts(dds) >= config$min_gene_copies) >= config$min_samples_per_cond
dds <- dds[keep_genes, ]
vst_counts <- vst(dds, blind = FALSE)

## 3. Dispersion Diagnostics
Calculates size factors and dispersion estimates, plotting the fit to ensure data quality and model appropriateness.

### Understanding DESeq2 Dispersion Plots

In RNA-Seq data, gene expression variance naturally scales with the mean (highly expressed genes show higher absolute variance). 
To model this accurately, DESeq2 estimates a **dispersion** parameter for each gene, which measures how much a gene's variance exceeds its mean.

A standard diagnostics plot visualises this relationship across three layers:
1. **Black Dots (`dispGeneEst`)**: The raw, independent dispersion estimate for each individual gene based purely on its sample variance.
2. **Red Line (`dispFit`)**: The global trend line fit across all genes, capturing how dispersion typically decreases as the mean expression increases.
3. **Blue Dots (`dispersion`)**: The final, shrunken dispersion values used for differential expression testing. DESeq2 shrinks gene-wise estimates toward the red line to minimise false positives from low-abundance genes, while allowing true biological outliers to remain high.

The horizontal flat "ceiling" of blue and black dots, present at the top-left corner of the plot, corresponds to low-expression genes with extremely high variance.
Because these genes have extremely low mean counts, they are expected to naturally be filtered out or mathematically penalised during differential expression testing due to their massive variance. They should not skew your downstream results or cause false positives. 

In [ ]:
dds <- estimateSizeFactors(dds, quiet = TRUE)
dds <- estimateDispersions(dds, quiet = TRUE)

disp_df <- data.frame(
    baseMean    = mcols(dds)$baseMean,
    dispGeneEst = mcols(dds)$dispGeneEst,
    dispersion  = mcols(dds)$dispersion,
    dispFit     = mcols(dds)$dispFit
) %>% filter(baseMean > 0)

p_disp <- ggplot(disp_df, aes(x = baseMean)) +
    geom_point(aes(y = dispGeneEst), color = "black", alpha = 0.3, size = 1) +
    geom_point(aes(y = dispersion), color = "dodgerblue", alpha = 0.3, size = 1, shape = 1) +
    geom_line(aes(y = dispFit), color = "red", linewidth = 1) +
    scale_x_log10() + scale_y_log10() +
    theme_minimal() +
    labs(title = "Dispersion Estimates", x = "Mean of Normalised Counts", y = "Dispersion")

print(p_disp)
ggsave("f_figures/Dispersion_Estimates.png", plot = p_disp, width = 8, height = 6, dpi = 300)

## 4. Dimensionality Reduction & Diversity Testing
Performs Principal Component Analysis (PCA) to check sample clustering, followed by PERMANOVA and Beta Dispersion tests.

**Expected Figure Trends:** The PCA plot is anticipated to display distinct and separate clusters representing the different `LabelingState` and `AssayType` groups along the primary principal components (PC1 and PC2). High biological and technical differentiation will be reflected by tight intra-group sample groupings and large inter-group distances.

In [ ]:
# Dynamically assign formatting based on available factors
if (design_type == 1) {
    pca_groups <- c("LabelingState")
    pca_palette <- c("lab" = "#009E73", "unlab" = "#CC79A7")
    pca_shapes <- c("lab" = 16, "unlab" = 15)
    group_labels <- c("lab" = "Labelled", "unlab" = "Unlabelled")
} else if (design_type == 2) {
    pca_groups <- c("AssayType")
    pca_palette <- c("input" = "#D55E00", "pull_down" = "#009E73")
    pca_shapes <- c("input" = 16, "pull_down" = 17)
    group_labels <- c("input" = "Input", "pull_down" = "Pull-Down")
} else if (design_type %in% c(3, 4)) {
    pca_groups <- c("LabelingState", "AssayType")
    pca_palette <- c("lab:input" = "#D55E00", "lab:pull_down" = "#009E73", "unlab:input" = "#56B4E9", "unlab:pull_down" = "#CC79A7")
    pca_shapes <- c("lab:input" = 16, "lab:pull_down" = 17, "unlab:input" = 15, "unlab:pull_down" = 18)
    group_labels <- c("lab:input" = "Labelled Input", "lab:pull_down" = "Labelled Pull-Down", "unlab:input" = "Unlabelled Input", "unlab:pull_down" = "Unlabelled Pull-Down")
}

pca_data <- plotPCA(vst_counts, intgroup = pca_groups, ntop = config$num_pca_genes, returnData = TRUE)
var_explained <- round(attr(pca_data, "percentVar") * 100, 1)

p_pca <- ggplot(pca_data, aes(x = PC1, y = PC2, color = group, shape = group)) +
    geom_point(size = 2.5) +
    scale_color_manual(name = "Group", values = pca_palette, labels = group_labels) +
    scale_shape_manual(name = "Group", values = pca_shapes, labels = group_labels) +
    labs(title = "Principal Component Analysis", 
         x = glue("PC1: {var_explained[1]}% variance"),
         y = glue("PC2: {var_explained[2]}% variance")) +
    theme_minimal() +
    theme(panel.border = element_rect(color = "black", fill = NA, linewidth = 0.5))

print(p_pca)

# Distance matrix for statistical testing
dist_mat <- vegdist(t(assay(vst_counts)), method = "euclidean")

# Define the full formula explicitly including the distance matrix on the left-hand side
if (design_type == 1) {
    perm_formula <- dist_mat ~ LabelingState
} else if (design_type == 2) {
    perm_formula <- dist_mat ~ AssayType
} else if (design_type == 4) {
    perm_formula <- dist_mat ~ Group
} else {
    perm_formula <- dist_mat ~ LabelingState * AssayType
}

cat("\n--- PERMANOVA ---\n")
print(adonis2(perm_formula, data = col_data, permutations = 999))

cat("\n--- Beta Dispersion ---\n")
if (design_type == 3) {
    print(anova(betadisper(dist_mat, group = interaction(col_data$LabelingState, col_data$AssayType))))
} else {
    print(anova(betadisper(dist_mat, group = col_data[[pca_groups[1]]])))
}

## 5. Differential Expression Analysis
Executes the DESeq model, extracts relevant contrasts using LFC shrinkage, and categorises genes into strict confidence tiers.

### Differential Expression Stratification & Tiered Categorisation

To isolate biological signals from technical background noise, the pipeline evaluates three distinct contrasts using DESeq2:

1. **`Lab_PD_vs_Input`**: Evaluates enrichment in the labelled pull-down sample against its paired labelled input background.
2. **`Lab_PD_vs_All_Input`**: Compares the labelled pull-down against a pooled average of both labelled and unlabelled inputs, providing a robust baseline control.
3. **`Lab_vs_Unlab_PD`**: Compares the labelled pull-down directly against the unlabelled pull-down to identify transcripts whose capture is strictly dependent on the presence of the metabolic label.

#### Confidence Tiering Criteria

Post-shrinkage log₂ fold changes ($L2FC$) and significance statistics (adjusted p-values/$qval$ and nominal p-values/$pval$) are combined to group genes into strict confidence tiers based on the directionality and strength of their expression shifts:

Our strong confidence criteria is based on: Subcellular transcriptome sequencing with single cell APEX-seq identifies regulators of cell-cell interactions
Andrew Xue, Bo Cai, Qian Xue, Nianping Liu, Xiaojie Qiu, Rogelio A. Hernández-López, Alice Y. Ting
bioRxiv 2026.03.17.712496; doi: https://doi.org/10.64898/2026.03.17.712496

* **Tier 1 (Strongest Confidence)**: Significant after multiple testing corrections with a substantial fold change.
    * *Up*: $\log_2\text{(Fold Change)} > 0.5$ and $\text{Adjusted p-value} < 0.05$
    * *Down*: $\log_2\text{(Fold Change)} < -0.5$ and $\text{Adjusted p-value} < 0.05$
* **Tier 2 (Moderate Confidence)**: Significant after multiple testing corrections but displaying a milder directional shift.
    * *Up*: $0.0 < \log_2\text{(Fold Change)} \le 0.5$ and $\text{Adjusted p-value} < 0.05$
    * *Down*: $-0.5 \le \log_2\text{(Fold Change)} < 0.0$ and $\text{Adjusted p-value} < 0.05$
* **Tier 3 (Weak Stats, High Magnitude)**: Fails multiple testing corrections but exhibits a sharp change in expression magnitude with a significant unadjusted p-value.
    * *Up*: $\log_2\text{(Fold Change)} > 0.5$ and $\text{Nominal p-value} < 0.05$
    * *Down*: $\log_2\text{(Fold Change)} < -0.5$ and $\text{Nominal p-value} < 0.05$
* **Tier 4 (Weakest Confidence)**: Fails multiple testing corrections and shows a mild directional shift, backed only by a significant unadjusted p-value.
    * *Up*: $0.0 < \log_2\text{(Fold Change)} \le 0.5$ and $\text{Nominal p-value} < 0.05$
    * *Down*: $-0.5 \le \log_2\text{(Fold Change)} < 0.0$ and $\text{Nominal p-value} < 0.05$
* **Not Significant**: Genes failing to meet the minimum threshold criteria ($\text{Nominal p-value} \ge 0.05$).

In [ ]:
dds <- DESeq(dds, quiet = TRUE)
coefs <- resultsNames(dds)

# Define the contrast logic dynamically
if (design_type == 1) {
    contrasts <- list(
        Lab_vs_Unlab = list(name = "LabelingState_lab_vs_unlab")
    )
} else if (design_type == 2) {
    contrasts <- list(
        Lab_PD_vs_Input = list(name = "AssayType_pull_down_vs_input")
    )
} else if (design_type == 3) {
    contrasts <- list(
        Lab_PD_vs_Input = list(contrast = list(c("AssayType_pull_down_vs_input", "LabelingStatelab.AssayTypepull_down"))),
        Lab_PD_vs_All_Input = list(contrast = ifelse(coefs == "AssayType_pull_down_vs_input", 1, ifelse(coefs == "LabelingStatelab.AssayTypepull_down", 0.5, 0))),
        Lab_vs_Unlab_PD = list(contrast = list(c("LabelingState_lab_vs_unlab", "LabelingStatelab.AssayTypepull_down")))
    )
} else if (design_type == 4) {
    contrasts <- list(
        Lab_PD_vs_Unlab_PD = list(contrast = c("Group", "lab_pull_down", "unlab_pull_down"))
    )
}

# Safely extract and shrink results, explicitly passing 'res' to preserve nominal p-values
de_results <- lapply(contrasts, function(c) {
    if ("name" %in% names(c)) {
        res <- results(dds, name = c$name)
        shrunk <- lfcShrink(dds, coef = c$name, res = res, type = "apeglm", quiet = TRUE)
    } else {
        res <- results(dds, contrast = c$contrast)
        coef_name <- resultsNames(dds)[grep("lab_pull_down_vs_unlab_pull_down", resultsNames(dds))]
        shrunk <- lfcShrink(dds, coef = coef_name, res = res, type = "apeglm", quiet = TRUE)
    }
    list(unshrunk = res, shrunk = shrunk)
})

categorise_genes <- function(unshrunk, shrunk, conf) {
    # Added baseMean here so the plotting function can use this unified dataframe
    df <- data.frame(
        baseMean = unshrunk$baseMean,
        log2FoldChange = shrunk$log2FoldChange, 
        padj = unshrunk$padj, 
        pvalue = unshrunk$pvalue, 
        row.names = rownames(shrunk)
    ) %>% drop_na()
    
    df <- df %>% mutate(
        Confidence_Tier = case_when(
            log2FoldChange > conf$log2_fc_thresh  & padj < conf$padj_thresh  ~ glue("Tier 1: L2FC>0.5, qval<0.05"),
            log2FoldChange > 0                    & padj < conf$padj_thresh  ~ "Tier 2: L2FC>0.0, qval<0.05",
            log2FoldChange > conf$log2_fc_thresh  & pvalue < conf$pval_thresh ~ "Tier 3: L2FC>0.5, pval<0.05",
            log2FoldChange > 0                    & pvalue < conf$pval_thresh ~ "Tier 4: L2FC>0.0, pval<0.05",
            log2FoldChange < -conf$log2_fc_thresh & padj < conf$padj_thresh  ~ "Tier 1: L2FC<-0.5, qval<0.05",
            log2FoldChange < 0                    & padj < conf$padj_thresh  ~ "Tier 2: L2FC<0.0, qval<0.05",
            log2FoldChange < -conf$log2_fc_thresh & pvalue < conf$pval_thresh ~ "Tier 3: L2FC<-0.5, pval<0.05",
            log2FoldChange < 0                    & pvalue < conf$pval_thresh ~ "Tier 4: L2FC<0.0, pval<0.05",
            TRUE ~ "Not Significant"
        )
    )
    
    tiers <- c("Tier 1: L2FC>0.5, qval<0.05", "Tier 2: L2FC>0.0, qval<0.05", "Tier 3: L2FC>0.5, pval<0.05", "Tier 4: L2FC>0.0, pval<0.05",
               "Tier 1: L2FC<-0.5, qval<0.05", "Tier 2: L2FC<0.0, qval<0.05", "Tier 3: L2FC<-0.5, pval<0.05", "Tier 4: L2FC<0.0, pval<0.05", "Not Significant")
    
    df$Confidence_Tier <- factor(df$Confidence_Tier, levels = tiers)
    return(df)
}

tiered_results <- lapply(names(de_results), function(name) {
    tier_df <- categorise_genes(de_results[[name]]$unshrunk, de_results[[name]]$shrunk, config)
    print(table(tier_df$Confidence_Tier))
    write.csv(tier_df, file = glue("o_outputs/processed_data/DE_results/Tiered_Results_{name}.csv"))
    tier_df
})
names(tiered_results) <- names(de_results)

## 6. Differential Expression Visualisation (Volcano & MA Plots)

This section maps a custom diagnostic plotting function across all three shrunken DESeq2 contrasts. 

For each comparison, the workflow systematically evaluates log₂ fold change and multiple-testing adjusted p-value boundaries, producing two standard visual outputs:
1. **Volcano Plots**: Dissect biological statistical significance against structural expression magnitude shifts, colour-coding significantly changing transcripts.
2. **MA Plots (Bland-Altman plots)**: Scale the distribution of shrunken log₂ fold changes against the mean of normalised counts to verify that variation balances correctly across low, moderate, and high expression thresholds.

Outputs are printed directly inline and automatically exported using the canonical contrast name identifier keys.

**Expected Figure Trends:**
* **Volcano Plots**: Expected to display a symmetric volcano-like dispersion where significant transcripts breach the defined p-value (horizontal) and log-fold change (vertical) limits, highlighted in red (up-regulated) or blue (down-regulated)
* **MA Plots**: Should confirm stable variance across the range of mean expression, tracking symmetrically around the zero log-fold change axis, demonstrating that significance isn't falsely driven by low-count noise.

In [ ]:
plot_diagnostics <- function(df, name, conf, ptype = "padj") {
    
    sig_col <- if(ptype == "pval") "pvalue" else "padj"
    thresh <- if(ptype == "pval") conf$pval_thresh else conf$padj_thresh
    
    df <- df %>% mutate(
        Significance = case_when(
            log2FoldChange > conf$log2_fc_thresh & .data[[sig_col]] < thresh ~ "Up-regulated",
            log2FoldChange < -conf$log2_fc_thresh & .data[[sig_col]] < thresh ~ "Down-regulated",
            TRUE ~ "Not Significant"
        )
    )
    
    y_metric <- -log10(df[[sig_col]])
    y_label  <- ifelse(ptype == "pval", "-Log10(Nominal P-value)", "-Log10(Adjusted P-value)")
    subtitle <- ifelse(ptype == "pval", "Weak Confidence (Nominal p-value)", "High Confidence (Adjusted q-value)")

    p_volc <- ggplot(df, aes(x = log2FoldChange, y = y_metric, color = Significance)) +
        geom_point(alpha = 0.6, size = 1.2) + 
        scale_color_manual(values = c("Up-regulated" = "red", "Down-regulated" = "blue", "Not Significant" = "grey")) +
        geom_vline(xintercept = c(-conf$log2_fc_thresh, conf$log2_fc_thresh), linetype = "dashed", color = "black") +
        geom_hline(yintercept = -log10(thresh), linetype = "dashed", color = "black") +
        theme_minimal() +
        theme(legend.position = "none") + 
        labs(title = "Volcano Plot", x = "Log2 Fold Change", y = y_label)
    
    p_ma <- ggplot(df, aes(x = baseMean, y = log2FoldChange, color = Significance)) +
        geom_point(alpha = 0.6, size = 1.2) +
        scale_x_log10(labels = scales::trans_format("log10", scales::math_format(10^.x))) +
        scale_color_manual(values = c("Up-regulated" = "red", "Down-regulated" = "blue", "Not Significant" = "grey")) +
        geom_hline(yintercept = 0, color = "black", linewidth = 0.5) +
        geom_hline(yintercept = c(-conf$log2_fc_thresh, conf$log2_fc_thresh), linetype = "dashed", color = "grey40") +
        coord_cartesian(ylim = c(-4, 4)) +
        theme_minimal() +
        labs(title = "MA Plot", x = "Mean of Normalised Counts", y = "Log2 Fold Change")
    
    combined <- (p_volc + p_ma) + 
        plot_layout(guides = "collect") + 
        plot_annotation(
            title = glue("Diagnostic Plots: {name}"),
            subtitle = glue("Confidence Level: {subtitle} | Thresholds: |L2FC| > {conf$log2_fc_thresh}, p < {thresh}"),
            theme = theme(plot.title = element_text(size = 14, face = "bold", hjust = 0.5), plot.subtitle = element_text(size = 11, face = "italic", color = "grey30", hjust = 0.5))
        )
    
    print(combined)
    ggsave(glue("f_figures/DE/Diagnostic_Panel_{name}_{ptype}.png"), plot = combined, width = 12, height = 5, dpi = 300)
}

# Now we feed the unified dataframe directly into the plotting function
invisible(lapply(names(tiered_results), function(n) plot_diagnostics(tiered_results[[n]], n, config, "padj")))
invisible(lapply(names(tiered_results), function(n) plot_diagnostics(tiered_results[[n]], n, config, "pval")))

## 7. Network Analysis (Gene Correlation)
Extracts the top dynamically changing transcripts to build a co-expression network map visualised via `ggplot2`.

**Expected Figure Trends:** The plotted heatmap is expected to highlight clustered expression profiles, identifying subsets of highly correlated genes that share co-expression patterns specifically induced in the labelled pull-down cohort compared to input constraints.

In [ ]:
norm_counts <- assay(vst_counts)
top_genes <- names(sort(apply(norm_counts, 1, var), decreasing = TRUE)[1:config$num_network_genes])

heat_df <- norm_counts[top_genes, ] %>%
    as.data.frame() %>%
    rownames_to_column("Gene") %>%
    pivot_longer(-Gene, names_to = "Sample", values_to = "Expression") %>%
    left_join(rownames_to_column(col_data, "Sample"), by = "Sample") %>%
    mutate(Sample = factor(Sample, levels = gtools::mixedsort(unique(Sample))))

p_heat <- ggplot(heat_df, aes(x = Sample, y = Gene, fill = Expression)) +
    geom_tile() +
    scale_fill_viridis_c(option = "magma", name = "VST Counts") + 
    labs(title = glue("Co-expression of Top {config$num_network_genes} Variable Genes"), x = "Samples", y = "Genes") +
    theme_minimal() +
    theme(axis.text.y = element_blank(), axis.text.x = element_text(angle = 45, hjust = 1), panel.grid = element_blank())

print(p_heat)
ggsave(glue("f_figures/Network-coexpr/Top_{config$num_network_genes}_Coexpression_Heatmap.png"), plot = p_heat, width = 8, height = 6, dpi = 300)

## 8. Gene Ontology (GO) Enrichment Analysis & Visualisation
Identifies enriched biological processes among target gene clusters and produces dotplot visualisations.

**Expected Figure Trends:** The generated dotplots will map enriched functional categories on the y-axis against the gene ratio on the x-axis. Larger dots represent higher target gene hit counts per pathway, with colour intensity signifying stronger adjusted p-values. It is anticipated that ciliary, metabolic, or translation-related terms will emerge in the strongly upregulated clusters.

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 12)

get_genes <- function(df, tiers) rownames(df[df$Confidence_Tier %in% tiers, ])

run_go <- function(genes, universe) {
    if (length(genes) == 0) return(NULL)
    enrichGO(gene = genes, universe = universe, OrgDb = org.Mm.eg.db, keyType = "SYMBOL", ont = "ALL", 
             pAdjustMethod = "BH", pvalueCutoff = 0.05, qvalueCutoff = 0.2, readable = TRUE)
}

format_cilia_info <- function(go_res) {
    if (is.null(go_res) || nrow(go_res) == 0) return("Cilium-related terms: 0")
    cilia <- go_res[grep("cili", go_res$Description, ignore.case = TRUE), ]
    if (nrow(cilia) > 0) {
        genes <- sort(unique(unlist(strsplit(paste(cilia$geneID, collapse = "/"), "/"))))
        return(glue("Cilium-related terms: {nrow(cilia)} | Unique Cilia Genes: {length(genes)} (e.g., {paste(head(genes, 3), collapse=', ')}...)"))
    }
    return("Cilium-related terms: 0")
}

build_go_plot <- function(go_obj, title, subtitle) {
    if (is.null(go_obj) || nrow(go_obj) == 0) return(ggplot() + theme_void() + labs(title = title, subtitle = "No enrichment found"))
    dotplot(go_obj, showCategory = 10) + 
        labs(title = title, subtitle = subtitle) +
        theme(axis.text.y = element_text(size = 9), axis.title.y = element_blank())
}

for (name in names(tiered_results)) {
    df <- tiered_results[[name]]
    univ <- rownames(df)
    
    go_up <- run_go(get_genes(df, c("Tier 1: L2FC>0.5, qval<0.05", "Tier 2: L2FC>0.0, qval<0.05", "Tier 3: L2FC>0.5, pval<0.05", "Tier 4: L2FC>0.0, pval<0.05")), univ)
    go_down <- run_go(get_genes(df, c("Tier 1: L2FC<-0.5, qval<0.05", "Tier 2: L2FC<0.0, qval<0.05")), univ)
    
    # Export Data
    if (!is.null(go_up)) write.csv(as.data.frame(go_up), glue("o_outputs/processed_data/GO_results/{name}_GO_StrongConf_UP.csv"))
    if (!is.null(go_down)) write.csv(as.data.frame(go_down), glue("o_outputs/processed_data/GO_results/{name}_GO_StrongConf_DOWN.csv"))
    
    # Combine into a subfigured patchwork layout
    p_up <- build_go_plot(go_up, "Up-regulated Targets", format_cilia_info(as.data.frame(go_up)))
    p_down <- build_go_plot(go_down, "Down-regulated Targets", format_cilia_info(as.data.frame(go_down)))
    
    combined_go <- (p_up / p_down) + 
        plot_annotation(title = glue("Strong Confidence GO Enrichment: {name}"), theme = theme(plot.title = element_text(face = "bold", size = 14)))
    
    print(combined_go)
    ggsave(glue("f_figures/GO/{name}_GO_StrongConf_Combined.png"), plot = combined_go, width = 10, height = 12, dpi = 300)
}

## 9. Gene Set Enrichment Analysis (GSEA)
Generates a ranked metric for genes to evaluate broader pathway activation/suppression, outputting global visualisations and targeted running scores.

**Expected Figure Trends:** The GSEA dotplot should partition overall biological themes structurally by direction of enrichment. The targeted mountain plot is expected to chart a pronounced running enrichment score that peaks towards the top or bottom extreme of the ranked gene list, indicating coordinate induction or suppression of the entire ciliary pathway module.

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 12)

for (name in names(de_results)) {
    res <- as.data.frame(de_results[[name]]$shrunk) %>% drop_na(pvalue, log2FoldChange)
    
    # metric <- sort(setNames(res$log2FoldChange / res$lfcSE, rownames(res)), decreasing = TRUE)

    set.seed(123)
    ranking_values <- sign(res$log2FoldChange) * -log10(res$pvalue) + rnorm(length(res$log2FoldChange), mean = 0, sd = 1e-8)
    metric <- sort(setNames(ranking_values, rownames(res)), decreasing = TRUE)

    gsea_res <- gseGO(geneList = metric, OrgDb = org.Mm.eg.db, keyType = "SYMBOL", ont = "ALL", 
                      pvalueCutoff = 0.1, pAdjustMethod = "BH", eps = 0, seed = 123)
    
    if (!is.null(gsea_res) && nrow(gsea_res) > 0) {
        gsea_df <- as.data.frame(gsea_res)
        write.csv(gsea_df, glue("o_outputs/processed_data/GO_results/{name}_GSEA_GO_Enrichment.csv"))
        
        cilia_terms <- gsea_df[grep("cili", gsea_df$Description, ignore.case = TRUE), ]
        sub_txt <- glue("GSEA Cilia Pathways Identified: {nrow(cilia_terms)}")
        
        p_gsea <- dotplot(gsea_res, showCategory = 15, split = ".sign") +
            facet_grid(. ~ .sign) +
            labs(title = glue("GSEA Functional Shifts: {name}"), subtitle = sub_txt) +
            theme(axis.text.y = element_text(size = 8))
        
        print(p_gsea)
        ggsave(glue("f_figures/GO/{name}_GSEA_Dotplot.png"), plot = p_gsea, width = 11, height = 8, dpi = 300)
        
        if (nrow(cilia_terms) > 0) {
            p_mount <- gseaplot2(gsea_res, geneSetID = cilia_terms$ID[1], title = glue("{name} Running Score: {cilia_terms$Description[1]}"))
            print(p_mount)
            ggsave(glue("f_figures/GO/{name}_GSEA_Mountain_Cilia.png"), plot = p_mount, width = 8, height = 6, dpi = 300)
        }
    }
}

## 10. Deep-Dive: Ciliary Basal Body Bi-directional Drivers
Extracts "leading edge" genes from the GSEA mountain plot to identify specific drivers of bi-directional pathway shifts. Visualises expression, builds isolated co-expression networks, and exports lists for motif analysis.

**Expected Figure Trends:** Replacing previous base-R implementations with `ggplot2` representations, the resulting heatmaps will explicitly map core sub-network components. The deep-dive expression map will stratify condition-dependent gene transcription variations, whereas the correlation heatmap will outline robustly co-regulated gene modules mapping the functional landscape of the queried biological pathways.

In [ ]:
for (name in names(de_results)) {
    gsea_file <- glue("o_outputs/processed_data/GO_results/{name}_GSEA_GO_Enrichment.csv")
    if (!file.exists(gsea_file)) next
    
    gsea_df <- read.csv(gsea_file, stringsAsFactors = FALSE)
    cilia_terms <- gsea_df[grep("cili", gsea_df$Description, ignore.case = TRUE), ]
    if (nrow(cilia_terms) == 0) next

    term_desc <- cilia_terms$Description[1]
    clean_term <- gsub(" ", "_", term_desc)
    
    genes <- unlist(strsplit(cilia_terms$core_enrichment[1], "/"))
    lfc_vals <- setNames(as.data.frame(de_results[[name]]$shrunk)[genes, "log2FoldChange"], genes)
    
    up_drv <- names(lfc_vals[!is.na(lfc_vals) & lfc_vals > 0])
    dn_drv <- names(lfc_vals[!is.na(lfc_vals) & lfc_vals < 0])
    
    write.table(up_drv, glue("o_outputs/processed_data/GO_results/{name}_{clean_term}_Up_Drivers.txt"), row.names = FALSE, col.names = FALSE, quote = FALSE)
    write.table(dn_drv, glue("o_outputs/processed_data/GO_results/{name}_{clean_term}_Down_Drivers.txt"), row.names = FALSE, col.names = FALSE, quote = FALSE)

    drv_counts <- norm_counts[intersect(genes, rownames(norm_counts)), ]
    
    expr_df <- as.data.frame(drv_counts) %>%
        rownames_to_column("Gene") %>%
        pivot_longer(-Gene, names_to = "Sample", values_to = "Expression") %>%
        left_join(rownames_to_column(col_data, "Sample"), by = "Sample") %>%
        mutate(Sample = factor(Sample, levels = gtools::mixedsort(unique(Sample))))

    p_expr <- ggplot(expr_df, aes(x = Sample, y = Gene, fill = Expression)) +
        geom_tile(color = "white") +
        scale_fill_viridis_c(option = "magma", name = "Counts") +
        theme_minimal() +
        labs(title = "Expression Sub-Network", x = "Sample", y = "Gene") +
        theme(axis.text.x = element_text(angle = 45, hjust = 1), axis.text.y = element_text(size = 6))

    cor_df <- as.data.frame(cor(t(drv_counts), method = "pearson")) %>%
        rownames_to_column("Gene1") %>%
        pivot_longer(-Gene1, names_to = "Gene2", values_to = "Correlation")

    p_cor <- ggplot(cor_df, aes(x = Gene1, y = Gene2, fill = Correlation)) +
        geom_tile(color = "white") +
        scale_fill_gradient2(low = "blue", high = "red", mid = "white", midpoint = 0, limit = c(-1, 1)) +
        theme_minimal() +
        labs(title = "Co-expression Matrix", x = "", y = "") +
        theme(axis.text.x = element_text(angle = 90, hjust = 1, vjust = 0.5, size = 6), axis.text.y = element_text(size = 6))

    # Combine deeply related plots using patchwork
    combined_deepdive <- (p_expr + p_cor) + 
        plot_annotation(
            title = glue("Deep-Dive Drivers [{name}]: {term_desc}"), 
            subtitle = glue("Total Leading Edge: {length(genes)} | Up-regulated: {length(up_drv)} | Down-regulated: {length(dn_drv)}"),
            theme = theme(plot.title = element_text(face = "bold", size = 14), plot.subtitle = element_text(color = "grey30"))
        )
        
    print(combined_deepdive)
    ggsave(glue("f_figures/GO/{name}_{clean_term}_DeepDive_Panel.png"), plot = combined_deepdive, width = 14, height = 8, dpi = 300)
}